In [46]:
from collections import Counter
import pandas as pd
import numpy as np
import scipy.stats as ss

In [3]:
# IN BOTH POCKET PAIRS AND TRIPLETS, CONSIDER POCKETVEC DISTANCES < 0.17 ###

In [4]:
# Get proteins
proteins = sorted(set(pd.read_csv("../data/mtb_trna_synthetases_bosch_2021_fig5_annotated.csv")['uniprot_ac']))

# Load relevant data
protein_to_gene = pd.read_csv("../data/mtb_trna_synthetases_bosch_2021_fig5_annotated.csv")
protein_to_gene = {i: j for i,j in zip(protein_to_gene['uniprot_ac'], protein_to_gene['gene_name_in_bosch_2021'])}

# Read Vulnerability Index from Bosch et al 2021
vulnerability = pd.read_excel("../data/bosch_2021_DataS2.xlsx", sheet_name="(1) Mtb H37Rv")
vulnerability = vulnerability[vulnerability['name'].isin(set(protein_to_gene.values()))].reset_index(drop=True)
gene_to_vulnerability = {i: j for i,j in zip(vulnerability['name'], vulnerability['Vulnerability Index'])}

# Load protein pair data
ALL_RESULTS_PAIRS = pd.read_csv("../processed/protein_prioritization/pairs_filtered.tsv", sep='\t', low_memory=False)
ALL_RESULTS_PAIRS["InterPro-Pocket1"] = ALL_RESULTS_PAIRS["InterPro-Pocket1"].fillna("").astype(str)
ALL_RESULTS_PAIRS["InterPro-Pocket2"] = ALL_RESULTS_PAIRS["InterPro-Pocket2"].fillna("").astype(str)

# Load protein triplet data
ALL_RESULTS_TRIPLETS = pd.read_csv("../processed/protein_prioritization/triplets_filtered.tsv", sep='\t', low_memory=False)
ALL_RESULTS_TRIPLETS["interpro_pocket1"] = ALL_RESULTS_TRIPLETS["interpro_pocket1"].fillna("").astype(str)
ALL_RESULTS_TRIPLETS["interpro_pocket2"] = ALL_RESULTS_TRIPLETS["interpro_pocket2"].fillna("").astype(str)
ALL_RESULTS_TRIPLETS["interpro_pocket3"] = ALL_RESULTS_TRIPLETS["interpro_pocket3"].fillna("").astype(str)

def get_number_pocket_pairs(df):
    return len(set(tuple(sorted([i, j])) for i, j in zip(df['Pocket1'], df['Pocket2'])))

def get_number_protein_pairs(df):
    return len(set(tuple(sorted([i, j])) for i, j in zip(df['Protein1'], df['Protein2'])))

def get_number_pocket_triplets(df):
    return len(set(tuple(sorted([i, j, k])) for i, j, k in zip(df['pocket1'], df['pocket2'], df['pocket3'])))

def get_number_protein_triplets(df):
    return len(set(tuple(sorted([i, j, k])) for i, j, k in zip(df['protein1'], df['protein2'], df['protein3'])))


In [5]:
# ### COUNTING CATALYTIC SITES AND GLOBAL SIMILARITY ###

# # Load protein pair data
# ALL_RESULTS_PAIRS = pd.read_csv("../processed/protein_prioritization/pairs.tsv", sep='\t', low_memory=False)
# ALL_RESULTS_PAIRS["InterPro-Pocket1"] = ALL_RESULTS_PAIRS["InterPro-Pocket1"].fillna("").astype(str)
# ALL_RESULTS_PAIRS["InterPro-Pocket2"] = ALL_RESULTS_PAIRS["InterPro-Pocket2"].fillna("").astype(str)

# # Load protein triplet data
# ALL_RESULTS_TRIPLETS = pd.read_csv("../processed/protein_prioritization/triplets.tsv", sep='\t', low_memory=False)
# ALL_RESULTS_TRIPLETS["interpro_pocket1"] = ALL_RESULTS_TRIPLETS["interpro_pocket1"].fillna("").astype(str)
# ALL_RESULTS_TRIPLETS["interpro_pocket2"] = ALL_RESULTS_TRIPLETS["interpro_pocket2"].fillna("").astype(str)
# ALL_RESULTS_TRIPLETS["interpro_pocket3"] = ALL_RESULTS_TRIPLETS["interpro_pocket3"].fillna("").astype(str)

# RMSD_CUT_OFF = 10
# SEQID_CUT_OFF = 35

# ### PAIRS ###

# # CATALYTIC COUNTS
# CATALYTIC_COUNTS = []
# for i,j in zip(ALL_RESULTS_PAIRS['InterPro-Pocket1'], ALL_RESULTS_PAIRS['InterPro-Pocket2']):
#     counts = []
#     for interpro in [i,j]:
#         if "Catalytic Domain (ATP Binding Site)" in interpro:
#             counts.append(1)
#         else:
#             counts.append(0)
#     CATALYTIC_COUNTS.append(sum(counts))
# ALL_RESULTS_PAIRS['catalytic_counts'] = CATALYTIC_COUNTS


# # GLOBAL SIMILARITY
# GLOBAL_SIMILARITY = []
# for seqid, rmsd in zip(ALL_RESULTS_PAIRS['Protein SEQ ID (NW)'], ALL_RESULTS_PAIRS['Protein RMSD']):
#     counts = []
#     if seqid > SEQID_CUT_OFF and rmsd < RMSD_CUT_OFF:
#         counts.append(1)
#     else:
#         counts.append(0)
#     GLOBAL_SIMILARITY.append(sum(counts))
# ALL_RESULTS_PAIRS['global_similarity'] = GLOBAL_SIMILARITY

# ### TRIPLETS ###

# # CATALYTIC COUNTS
# CATALYTIC_COUNTS = []
# for i,j,k in zip(ALL_RESULTS_TRIPLETS['interpro_pocket1'], ALL_RESULTS_TRIPLETS['interpro_pocket2'], ALL_RESULTS_TRIPLETS['interpro_pocket3']):
#     counts = []
#     for interpro in [i,k,j]:
#         if "Catalytic Domain (ATP Binding Site)" in interpro:
#             counts.append(1)
#         else:
#             counts.append(0)
#     CATALYTIC_COUNTS.append(sum(counts))
# ALL_RESULTS_TRIPLETS['catalytic_counts'] = CATALYTIC_COUNTS


# # GLOBAL SIMILARITY
# GLOBAL_SIMILARITY = []
# for seqidA, seqidB, seqidC, rmsdA, rmsdB, rmsdC in zip(ALL_RESULTS_TRIPLETS['SEQ_ID_NW_A'], ALL_RESULTS_TRIPLETS['SEQ_ID_NW_B'], ALL_RESULTS_TRIPLETS['SEQ_ID_NW_C'], ALL_RESULTS_TRIPLETS['RMSD_A'], ALL_RESULTS_TRIPLETS['RMSD_B'], ALL_RESULTS_TRIPLETS['RMSD_C']):
#     counts = []
#     if seqidA > SEQID_CUT_OFF and rmsdA < RMSD_CUT_OFF:
#         counts.append(1)
#     else:
#         counts.append(0)
#     if seqidB > SEQID_CUT_OFF and rmsdB < RMSD_CUT_OFF:
#         counts.append(1)
#     else:
#         counts.append(0)
#     if seqidC > SEQID_CUT_OFF and rmsdC < RMSD_CUT_OFF:
#         counts.append(1)
#     else:
#         counts.append(0)
#     GLOBAL_SIMILARITY.append(sum(counts))
# ALL_RESULTS_TRIPLETS['global_similarity'] = GLOBAL_SIMILARITY

In [6]:
len(ALL_RESULTS_PAIRS), len(ALL_RESULTS_TRIPLETS)

(1481, 3880)

In [7]:
RESULTS = []
PROTEIN_TO_OCCURRENCES_PAIRS = {i: {} for i in proteins}
PROTEIN_TO_OCCURRENCES_TRIPLETS = {i: {} for i in proteins}
Cs_PAIRS = [[0,0], [0,1], [0,2], [1,0], [1,1], [1,2]]
Cs_TRIPLETS = [[0,0], [0,1], [0,2], [1,0], [1,1], [1,2], [2,0], [2,1], [2,2]]

# For each protein
for protein in proteins:

    # Mappings
    gene = protein_to_gene[protein]
    vul = gene_to_vulnerability[gene]

    # Get counts in pairs
    pair_counts = []
    for GS, CAT in Cs_PAIRS:
        if CAT == 2:
            COND = (ALL_RESULTS_PAIRS['global_similarity'] == GS) & (ALL_RESULTS_PAIRS['catalytic_counts'] >= CAT)
        else:
            COND = (ALL_RESULTS_PAIRS['global_similarity'] == GS) & (ALL_RESULTS_PAIRS['catalytic_counts'] == CAT)
        COND = ALL_RESULTS_PAIRS[COND].reset_index(drop=True)
        counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())[protein]
        pair_counts.append(counter)
        if protein == 'P9WFS9':
            print(f"PAIRS --- GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")

    # Get counts in triplets
    triplet_counts = []
    for GS, CAT in Cs_TRIPLETS:
        if CAT == 2:
            COND = (ALL_RESULTS_TRIPLETS['global_similarity'] == GS) & (ALL_RESULTS_TRIPLETS['catalytic_counts'] >= CAT)
        else:
            COND = (ALL_RESULTS_TRIPLETS['global_similarity'] == GS) & (ALL_RESULTS_TRIPLETS['catalytic_counts'] == CAT)
        COND = ALL_RESULTS_TRIPLETS[COND].reset_index(drop=True)
        counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())[protein]
        triplet_counts.append(counter)
        if protein == 'P9WFS9':
            print(f"TRIPLETS --- GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
        
    # Append everything to results
    RESULTS.append([protein, gene, vul] + pair_counts + triplet_counts)

columns = ["Protein", 'Gene name', 'Vulnerability score'] + [f"P_GS-{i}_CAT-{j}" for i,j in Cs_PAIRS] + [f"T_GS-{i}_CAT-{j}" for i,j in Cs_TRIPLETS]
RESULTS = pd.DataFrame(RESULTS, columns=columns)
RESULTS.to_csv("../processed/protein_prioritization/summary_counts_lenient.tsv", index=False, sep='\t')

PAIRS --- GS 0 -- CAT 0 -- 94[35]
PAIRS --- GS 0 -- CAT 1 -- 574[105]
PAIRS --- GS 0 -- CAT 2 -- 787[59]
PAIRS --- GS 1 -- CAT 0 -- 3[2]
PAIRS --- GS 1 -- CAT 1 -- 23[4]
PAIRS --- GS 1 -- CAT 2 -- 0[0]
TRIPLETS --- GS 0 -- CAT 0 -- 24[16]
TRIPLETS --- GS 0 -- CAT 1 -- 284[109]
TRIPLETS --- GS 0 -- CAT 2 -- 3,496[291]
TRIPLETS --- GS 1 -- CAT 0 -- 1[1]
TRIPLETS --- GS 1 -- CAT 1 -- 17[10]
TRIPLETS --- GS 1 -- CAT 2 -- 56[15]
TRIPLETS --- GS 2 -- CAT 0 -- 2[1]
TRIPLETS --- GS 2 -- CAT 1 -- 0[0]
TRIPLETS --- GS 2 -- CAT 2 -- 0[0]


In [8]:
### STRINGENT CASE ###

### IN BOTH POCKET PAIRS AND TRIPLETS, CONSIDER POCKETVEC DISTANCES < 0.14 (MIN 1/3 < 0.14 IN TRIPLETS)

In [9]:
# Get proteins
proteins = sorted(set(pd.read_csv("../data/mtb_trna_synthetases_bosch_2021_fig5_annotated.csv")['uniprot_ac']))

# Load relevant data
protein_to_gene = pd.read_csv("../data/mtb_trna_synthetases_bosch_2021_fig5_annotated.csv")
protein_to_gene = {i: j for i,j in zip(protein_to_gene['uniprot_ac'], protein_to_gene['gene_name_in_bosch_2021'])}

# Read Vulnerability Index from Bosch et al 2021
vulnerability = pd.read_excel("../data/bosch_2021_DataS2.xlsx", sheet_name="(1) Mtb H37Rv")
vulnerability = vulnerability[vulnerability['name'].isin(set(protein_to_gene.values()))].reset_index(drop=True)
gene_to_vulnerability = {i: j for i,j in zip(vulnerability['name'], vulnerability['Vulnerability Index'])}

# Load protein pair data
ALL_RESULTS_PAIRS = pd.read_csv("../processed/protein_prioritization/pairs_filtered.tsv", sep='\t', low_memory=False)
ALL_RESULTS_PAIRS["InterPro-Pocket1"] = ALL_RESULTS_PAIRS["InterPro-Pocket1"].fillna("").astype(str)
ALL_RESULTS_PAIRS["InterPro-Pocket2"] = ALL_RESULTS_PAIRS["InterPro-Pocket2"].fillna("").astype(str)

# Load protein triplet data
ALL_RESULTS_TRIPLETS = pd.read_csv("../processed/protein_prioritization/triplets_filtered.tsv", sep='\t', low_memory=False)
ALL_RESULTS_TRIPLETS["interpro_pocket1"] = ALL_RESULTS_TRIPLETS["interpro_pocket1"].fillna("").astype(str)
ALL_RESULTS_TRIPLETS["interpro_pocket2"] = ALL_RESULTS_TRIPLETS["interpro_pocket2"].fillna("").astype(str)
ALL_RESULTS_TRIPLETS["interpro_pocket3"] = ALL_RESULTS_TRIPLETS["interpro_pocket3"].fillna("").astype(str)

# Stringent filter: PocketVec distance and P2rank score
ALL_RESULTS_PAIRS = ALL_RESULTS_PAIRS[ALL_RESULTS_PAIRS['PocketVec distance'] < 0.14].reset_index(drop=True)
ALL_RESULTS_PAIRS = ALL_RESULTS_PAIRS[(ALL_RESULTS_PAIRS['P2Rank score 1'] > 5) & (ALL_RESULTS_PAIRS['P2Rank score 2'] > 5)].reset_index(drop=True)

ALL_RESULTS_TRIPLETS = ALL_RESULTS_TRIPLETS[(ALL_RESULTS_TRIPLETS['dist_A'] < 0.14) |
                                            (ALL_RESULTS_TRIPLETS['dist_B'] < 0.14) |
                                            (ALL_RESULTS_TRIPLETS['dist_C'] < 0.14)].reset_index(drop=True)

ALL_RESULTS_TRIPLETS = ALL_RESULTS_TRIPLETS[(ALL_RESULTS_TRIPLETS['P2Rank score 1'] > 5) &
                                            (ALL_RESULTS_TRIPLETS['P2Rank score 2'] > 5) &
                                            (ALL_RESULTS_TRIPLETS['P2Rank score 3'] > 5)].reset_index(drop=True)

In [10]:
len(ALL_RESULTS_PAIRS), len(ALL_RESULTS_TRIPLETS)

(76, 807)

In [11]:
RESULTS = []
PROTEIN_TO_OCCURRENCES_PAIRS = {i: {} for i in proteins}
PROTEIN_TO_OCCURRENCES_TRIPLETS = {i: {} for i in proteins}
Cs_PAIRS = [[0,0], [0,1], [0,2], [1,0], [1,1], [1,2]]
Cs_TRIPLETS = [[0,0], [0,1], [0,2], [1,0], [1,1], [1,2], [2,0], [2,1], [2,2]]

# For each protein
for protein in proteins:

    # Mappings
    gene = protein_to_gene[protein]
    vul = gene_to_vulnerability[gene]

    # Get counts in pairs
    pair_counts = []
    for GS, CAT in Cs_PAIRS:
        if CAT == 2:
            COND = (ALL_RESULTS_PAIRS['global_similarity'] == GS) & (ALL_RESULTS_PAIRS['catalytic_counts'] >= CAT)
        else:
            COND = (ALL_RESULTS_PAIRS['global_similarity'] == GS) & (ALL_RESULTS_PAIRS['catalytic_counts'] == CAT)
        COND = ALL_RESULTS_PAIRS[COND].reset_index(drop=True)
        counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())[protein]
        pair_counts.append(counter)
        if protein == 'P9WFS9':
            print(f"PAIRS --- GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")

    # Get counts in triplets
    triplet_counts = []
    for GS, CAT in Cs_TRIPLETS:
        if CAT == 2:
            COND = (ALL_RESULTS_TRIPLETS['global_similarity'] == GS) & (ALL_RESULTS_TRIPLETS['catalytic_counts'] >= CAT)
        else:
            COND = (ALL_RESULTS_TRIPLETS['global_similarity'] == GS) & (ALL_RESULTS_TRIPLETS['catalytic_counts'] == CAT)
        COND = ALL_RESULTS_TRIPLETS[COND].reset_index(drop=True)
        counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())[protein]
        triplet_counts.append(counter)
        if protein == 'P9WFS9':
            print(f"TRIPLETS --- GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
        
    # Append everything to results
    RESULTS.append([protein, gene, vul] + pair_counts + triplet_counts)

columns = ["Protein", 'Gene name', 'Vulnerability score'] + [f"P_GS-{i}_CAT-{j}" for i,j in Cs_PAIRS] + [f"T_GS-{i}_CAT-{j}" for i,j in Cs_TRIPLETS]
RESULTS = pd.DataFrame(RESULTS, columns=columns)
RESULTS.to_csv("../processed/protein_prioritization/summary_counts_stringent.tsv", index=False, sep='\t')

PAIRS --- GS 0 -- CAT 0 -- 0[0]
PAIRS --- GS 0 -- CAT 1 -- 11[9]
PAIRS --- GS 0 -- CAT 2 -- 64[22]
PAIRS --- GS 1 -- CAT 0 -- 0[0]
PAIRS --- GS 1 -- CAT 1 -- 1[1]
PAIRS --- GS 1 -- CAT 2 -- 0[0]
TRIPLETS --- GS 0 -- CAT 0 -- 0[0]
TRIPLETS --- GS 0 -- CAT 1 -- 3[3]
TRIPLETS --- GS 0 -- CAT 2 -- 802[95]
TRIPLETS --- GS 1 -- CAT 0 -- 0[0]
TRIPLETS --- GS 1 -- CAT 1 -- 1[1]
TRIPLETS --- GS 1 -- CAT 2 -- 1[1]
TRIPLETS --- GS 2 -- CAT 0 -- 0[0]
TRIPLETS --- GS 2 -- CAT 1 -- 0[0]
TRIPLETS --- GS 2 -- CAT 2 -- 0[0]


In [12]:
# RESULTS[[f"P_GS-{i}_CAT-{j}" for i,j in Cs_PAIRS]].values.flatten().sum() / 2
# RESULTS[[f"T_GS-{i}_CAT-{j}" for i,j in Cs_TRIPLETS]].values.flatten().sum() / 3

In [41]:
### PREPARE FINAL FILE ###

# Load Lenient and Stringent datasets
LENIENT = pd.read_csv("../processed/protein_prioritization/summary_counts_lenient.tsv", sep='\t')
STRINGENT = pd.read_csv("../processed/protein_prioritization/summary_counts_stringent.tsv", sep='\t')

# Get columns
cols1 = LENIENT.columns[:3]
cols2 = LENIENT.columns[3:]

# Get values
values_lenient = LENIENT[cols2]
values_stringent = STRINGENT[cols2]

In [132]:
ranks = []

# For each column (6+9) (*2)
for col in cols2:

    # Get ranks for both lenient and stringent
    r_lenient = np.array([-i for i in values_lenient[col].tolist()])
    r_lenient = ss.rankdata(r_lenient, method='max')
    ranks.append(r_lenient)
    r_stringent = np.array([-i for i in values_stringent[col].tolist()])
    r_stringent = ss.rankdata(r_stringent, method='max')
    ranks.append(r_stringent)

# Transpose matrix
ranks = np.column_stack(ranks)
ranks_all = [",".join(i.astype(str)) for i in ranks]

# Get averahe ranks
ranks = np.average(ranks, axis=1)

# Create final df
final_df = LENIENT[cols1].copy()
final_df['Average rank'] = ranks
# final_df['All ranks'] = ranks_all
final_df = final_df.sort_values('Average rank')

# Save it
final_df.to_csv("../processed/protein_prioritization/final_results.tsv", sep='\t', index=False)

In [127]:
np.average(np.array(final_df[final_df['Protein'] == "P9WFU3"]['All ranks'].tolist()[0].split(",")).astype(float))

np.float64(13.2)